
Lección 1: librería numpy

In [1]:
# link a github https://github.com/lucasalechilet/Curso_FCD/blob/main/Proyecto%20Mod%203/Proyecto_Mod_3.ipynb

import numpy as np
import pandas as pd


def generar_datos_numpy():
    # Generamos 300 ids de clientes y montos de transacciones aleatorios
    ids = np.arange(0, 300)
    #por simplicidad, usaremos la distribución uniforme para las transacciones
    mtx = np.random.uniform(1000, 100000, size=300)
    mtx[42]=200000 #cambiamos uno de los valores de transacción para generar un outlier para usar más adelante

    mtx=mtx.round(0) #redondeamos a 0 decimales para que los montos sean enteros

    # aplicamos las operaciones aritméticas que podrían resultar más útiles al estudiar mercados, el promedio de compra y el total de transacciones
    c_ids=ids.size
    c_mtx=ids.size
    media = np.mean(mtx)
    total = np.sum(mtx)

    #hacemos print para verificar que el código funcione correctamente
    print("Número ids: ",c_ids,"\n","Número transacciones: ", c_mtx,"\n","Monto Total transacciones: ", total,"\n","Promedio de compra: ", media)

    # hacemos return para usar esto en lo que continúa del proyecto
    return ids, mtx
ids,mtx=generar_datos_numpy()


Número ids:  300 
 Número transacciones:  300 
 Monto Total transacciones:  15272616.0 
 Promedio de compra:  50908.72


por qué numpy es eficiente en el manejo de datos numéricos:  
Hay 3 razones por las que numpy es tan eficiente:  
  
La primera es, al tener homogeneidad en los arrays que maneja, se sabe cada cuántos bits hay un elemento del array y no tiene que confirmar para cada elemento cuándo termina.

La segunda es que hace un procesamiento paralelo de las operaciones requeridas, lo que le permite hacer todo en menos tiempo, en lugar de hacerlo secuencial.  

La tercera razón es que esta librería integra código en C, C++ y Fortran, lo que es más eficiente que python.


Lección 2: librería pandas


In [2]:
def crear_dataframe(ids, mtx):
    #en base a datos del punto anterior, generamos el dataframe correspondiente
    df = pd.DataFrame({'id cliente': ids, 'monto transacción': mtx})

    #hacemos una exploración inicial de los datos:
    #partimos mostrando los primeros elementos y los últimos elementos.
    print("primeras filas:\n", df.head())
    print("últimas filas:\n", df.tail())

    #acá hacemos un print de los datos de estadísticas descriptivas usando describe
    print("\n estadísticas:\n", df.describe())

    #generamos un filtro condicional para las transacciones de montos sobre 95000
    tx_grandes = df[df['monto transacción'] > 95000]
    print(tx_grandes)

    # tomamos algunas filas y las concatenamos al DataFrame para tener duplicados que eliminar en la sección 5
    filas_duplicadas = df.iloc[[5, 23, 47, 80, 112]].copy()
    df = pd.concat([df, filas_duplicadas], ignore_index=True)

    return df

#ahora que tenemos la función definida, la podemos llamar con los datos que generamos en el bloque anterior
df=crear_dataframe(ids,mtx)

#finalmente, generamos el csv con estos datos
df.to_csv('df_preliminares.csv', index=False)

primeras filas:
    id cliente  monto transacción
0           0            69437.0
1           1            73888.0
2           2            46735.0
3           3            54367.0
4           4            29411.0
últimas filas:
      id cliente  monto transacción
295         295            52611.0
296         296             3001.0
297         297            39147.0
298         298            43256.0
299         299            10828.0

 estadísticas:
        id cliente  monto transacción
count  300.000000         300.000000
mean   149.500000       50908.720000
std     86.746758       27939.635926
min      0.000000        1307.000000
25%     74.750000       30264.500000
50%    149.500000       50169.500000
75%    224.250000       72651.250000
max    299.000000      200000.000000
     id cliente  monto transacción
42           42           200000.0
121         121            96812.0
139         139            97955.0
143         143            98498.0
146         146            97940.0

el uso de pandas es muy útil porque permite el procesamiento y filtrado de datos de forma rápida y con fácil sintaxis. También, tiene métodos como describe, que nos permiten generar un resumen sobre los datos de manera simple.

Lección 3: Obtención de datos desde archivos

In [3]:
def integrar_fuentes(df_base):
    # cargamos el csv generado en la lección 2
    df_csv = pd.read_csv('df_preliminares.csv')
    print("CSV cargado correctamente:\n", df_csv.head())

    # leemos la tabla web que fue generada externamente, agregada a mi github
    url_tabla = 'https://lucasalechilet.github.io/Curso_FCD/Proyecto%20Mod%203/tabla_regiones.html'

    tablas_web = pd.read_html(url_tabla) # read_html devuelve una lista de tablas
    df_web = tablas_web[0]  #tomamos la primera (única)

    # renombramos la columna 'id cliente' del csv base para hacer el merge
    # usamos rename para alinear los nombres de columna entre ambas fuentes
    df_csv = df_csv.rename(columns={'id cliente': 'cliente_id'})

    # combinamos las fuentes mediante merge() usando cliente_id como clave
    df_unificado = pd.merge(df_csv, df_web, on='cliente_id', how='left')
    print("\n DataFrame unificado (primeras filas):\n", df_unificado.head())
    print("\n Shape del DataFrame unificado:", df_unificado.shape)

    # guardamos el DataFrame unificado para usarlo en la siguiente sección
    df_unificado.to_csv('df_unificado.csv', index=False)

    return df_unificado

# llamamos a la función con el dataframe generado en la lección 2
df_unificado = integrar_fuentes(df)

CSV cargado correctamente:
    id cliente  monto transacción
0           0            69437.0
1           1            73888.0
2           2            46735.0
3           3            54367.0
4           4            29411.0

 DataFrame unificado (primeras filas):
    cliente_id  monto transacción region  puntos_fidelidad
0           0            69437.0  Norte             120.0
1           1            73888.0    Sur             340.0
2           2            46735.0   Este              87.0
3           3            54367.0  Oeste             456.0
4           4            29411.0  Norte             210.0

 Shape del DataFrame unificado: (305, 4)


Desafíos encontrados al integrar múltiples fuentes de datos:  

El primer desafío fue la inconsistencia en los nombres de columnas entre fuentes: el CSV usaba 'id cliente' (con espacio) y la tabla web usaba 'cliente_id', lo que requirió un rename() antes del merge.  
El segundo desafío fue lograr que se cargara la tabla de html correctamente, y tuve que cambiar la configuración de mi github para que funcione como web, y añadir el path correcto.
El tercer desafío fue elegir el tipo de merge: usamos 'left' para conservar todos los clientes del CSV base aunque no tengan información de región, evitando pérdida de registros.


Lección 4: Manejo de valores perdidos y outliers

In [4]:
def limpiar_datos(df):
    # primero hacemos una copia para no modificar el DataFrame original
    df = df.copy()

    # renombramos la columna de montos para mayor claridad en el resto del flujo
    if 'monto transacción' in df.columns:
        df = df.rename(columns={'monto transacción': 'monto'})

    # identificación de valores nulos
    print("Valores nulos por columna antes de limpiar:")
    print(df.isnull().sum())

    # simulamos valores nulos en la columna monto (filas 0 a 5)
    df.loc[0:5, 'monto'] = np.nan

    # imputamos los nulos con la media de la columna
    # esta decisión conserva el volumen de datos y minimiza el impacto en la distribución
    print("\n Valores nulos en 'monto' antes de imputación:", df['monto'].isnull().sum())
    df['monto'] = df['monto'].fillna(df['monto'].mean())
    print("\n Valores nulos en 'monto' luego de imputación:", df['monto'].isnull().sum())

    # para columnas categóricas como 'region', imputamos con la moda
    if 'region' in df.columns:
        moda_region = df['region'].mode()[0]
        df['region'] = df['region'].fillna(moda_region)
        print("Valores nulos en 'region' luego de imputación:", df['region'].isnull().sum())

    # detección y tratamiento de outliers con IQR
    Q1 = df['monto'].quantile(0.25)
    Q3 = df['monto'].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    # mostramos cuántos outliers se detectaron antes de filtrar
    outliers = df[(df['monto'] < limite_inferior) | (df['monto'] > limite_superior)]
    print("\n Outliers detectados por IQR:", len(outliers))
    print("Límite inferior:", round(limite_inferior, 2), "| Límite superior:", round(limite_superior, 2))

    # eliminamos los outliers del DataFrame
    df_limpio = df[(df['monto'] >= limite_inferior) & (df['monto'] <= limite_superior)]
    print("\n Registros antes de limpiar outliers:", len(df))
    print(" Registros después de limpiar outliers:", len(df_limpio))

    # guardamos el DataFrame limpio para usarlo en la siguiente sección
    df_limpio.to_csv('df_limpio.csv', index=False)
    print("\nDataFrame limpio guardado como df_limpio.csv")

    return df_limpio

# llamamos a la función con el DataFrame unificado de la sección 3
df_limpio = limpiar_datos(df_unificado)

Valores nulos por columna antes de limpiar:
cliente_id            0
monto                 0
region              201
puntos_fidelidad    201
dtype: int64

 Valores nulos en 'monto' antes de imputación: 6

 Valores nulos en 'monto' luego de imputación: 0
Valores nulos en 'region' luego de imputación: 0

 Outliers detectados por IQR: 1
Límite inferior: -31164.5 | Límite superior: 133183.5

 Registros antes de limpiar outliers: 305
 Registros después de limpiar outliers: 304

DataFrame limpio guardado como df_limpio.csv


Para los valores nulos en 'monto' se decidió imputar con la media en lugar de eliminar filas, ya que eliminar filas reduce el volumen de datos disponibles y podría sesgar el análisis si los nulos no son completamente aleatorios.  

Para la columna 'region' (categórica) se usó la moda, que es la estrategia más apropiada para variables no numéricas ya que no tiene sentido calcular una media sobre categorías.  

Para los outliers se utilizó el método IQR, que es robusto ante distribuciones no normales. Se decidió eliminarlos porque en un contexto de e-commerce, transacciones fuera del rango esperado suelen representar errores de registro más que operaciones reales. El único outlier fue el introducido artificialmente en la sección 1.

Lección 5: Data Wrangling

In [5]:
def data_wrangling(df):
    # hacemos una copia para no modificar el DataFrame de entrada
    df = df.copy()

    #eliminamos duplicados
    registros_antes = len(df)
    df = df.drop_duplicates()
    print("Duplicados eliminados:", registros_antes - len(df))

    #transformamos tipos de datos
    # aseguramos que cliente_id sea entero y monto sea int
    df['cliente_id'] = df['cliente_id'].astype(int)
    df['monto'] = df['monto'].astype(int)
    print("\n Tipos de datos luego de transformación:\n", df.dtypes)

    # creamos nuevas columnas calculadas
    # agregamos el monto con IVA (19%) usando apply() con lambda
    df['monto_con_iva'] = df['monto'].apply(lambda x: round(x * 1.19, 0))

    # creamos una columna que clasifica cada transacción según si supera la media
    media_monto = df['monto'].mean()
    df['sobre_promedio'] = df['monto'].map(lambda x: 'Sí' if x > media_monto else 'No')

    # discretizamos la columna monto en categorías de gasto
    # usamos pd.cut para dividir en 3 rangos iguales y etiquetarlos
    df['categoria_gasto'] = pd.cut(df['monto'], bins=3, labels=['Bajo', 'Medio', 'Alto'])
    print("\n Distribución por categoría de gasto:\n", df['categoria_gasto'].value_counts())

    # hacemos print de una muestra del DataFrame optimizado
    print("\n DataFrame optimizado (primeras filas):\n", df.head())

    # guardamos la versión optimizada
    df.to_csv('df_optimizado.csv', index=False)
    print("\n DataFrame optimizado guardado como df_optimizado.csv")

    return df

# llamamos a la función con el DataFrame limpio de la lección 4
df_optimizado = data_wrangling(df_limpio)

Duplicados eliminados: 4

 Tipos de datos luego de transformación:
 cliente_id            int64
monto                 int64
region               object
puntos_fidelidad    float64
dtype: object

 Distribución por categoría de gasto:
 categoria_gasto
Medio    114
Alto      96
Bajo      90
Name: count, dtype: int64

 DataFrame optimizado (primeras filas):
    cliente_id  monto region  puntos_fidelidad  monto_con_iva sobre_promedio  \
0           0  50649  Norte             120.0        60272.0             Sí   
1           1  50649    Sur             340.0        60272.0             Sí   
2           2  50649   Este              87.0        60272.0             Sí   
3           3  50649  Oeste             456.0        60272.0             Sí   
4           4  50649  Norte             210.0        60272.0             Sí   

  categoria_gasto  
0           Medio  
1           Medio  
2           Medio  
3           Medio  
4           Medio  

 DataFrame optimizado guardado como df_optimiza

El data wrangling permitió añadir al dataset columnas que podrían resultar útiles para el análisis: el monto con IVA facilita reportes financieros reales, la columna 'sobre_promedio' permite segmentar rápidamente a los clientes de mayor valor, y segmentar por montos añade una columna categórica que es más útil para agrupamientos y visualizaciones posteriores.

Lección 6: Agrupamiento y Pivoteo de Datos

In [6]:
def estructurar_final(df):
    # hacemos una copia para no modificar el DataFrame de entrada
    df = df.copy()

    # agrupamiento con groupby()
    # calculamos métricas resumidas de monto por región
    # usamos observed=True para evitar el FutureWarning de pandas con columnas categóricas
    if 'region' in df.columns:
        resumen_region = df.groupby('region', observed=True)['monto'].agg(['mean', 'sum', 'count'])
        resumen_region.columns = ['promedio_monto', 'total_monto', 'cantidad_tx']
        print("Resumen por región:\n", resumen_region)
    else:
        print("Columna 'region' no disponible, se omite el agrupamiento por región")

    # agrupamiento por categoría de gasto
    # observed=True evita incluir categorías vacías en el resultado
    resumen_categoria = df.groupby('categoria_gasto', observed=True)['monto'].agg(['mean', 'sum', 'count'])
    resumen_categoria.columns = ['promedio_monto', 'total_monto', 'cantidad_tx']
    print("\n Resumen por categoría de gasto:\n", resumen_categoria)

    # tabla pivote: filas=region, columnas=categoria_gasto, valores=cantidad de transacciones
    # observed=True también aplica aquí para no mostrar combinaciones vacías
    if 'region' in df.columns:
        pivot_tabla = df.pivot_table(
            index='region',
            columns='categoria_gasto',
            values='monto',
            aggfunc='count',
            observed=True
        )
        print("\n Tabla pivote (transacciones por región y categoría):\n", pivot_tabla)

    # uso de melt() para reestructurar en formato largo
    # tomamos un subconjunto para demostrar melt
    df_melt_base = df[['cliente_id', 'monto', 'monto_con_iva']].head(10)
    df_largo = df_melt_base.melt(
        id_vars='cliente_id',
        value_vars=['monto', 'monto_con_iva'],
        var_name='tipo_monto',
        value_name='valor'
    )
    print("\n Formato largo con melt() (muestra):\n", df_largo.head(8))

    # combinamos una fuente adicional con concat()
    # simulamos un segundo lote de transacciones para demostrar concat
    ids_extra = np.arange(300, 320)
    mtx_extra = np.random.uniform(1000, 100000, size=20)
    mtx_extra = mtx_extra.round(0)
    df_extra = pd.DataFrame({
        'cliente_id': ids_extra,
        'monto': mtx_extra,
        'monto_con_iva': mtx_extra * 1.19,
        'sobre_promedio': 'No',
        'categoria_gasto': pd.cut(mtx_extra, bins=3, labels=['Bajo', 'Medio', 'Alto'])
    })
    df_final = pd.concat([df, df_extra], ignore_index=True)
    print("\n Registros tras concat() con lote extra:", len(df_final))

    # exportamos el dataset final en CSV y Excel
    df_final.to_csv('dataset_final_ecommerce.csv', index=False)
    df_final.to_excel('dataset_final_ecommerce.xlsx', index=False)

    print("\n--- PROCESO FINALIZADO ---")
    print("Dataset exportado exitosamente en CSV y Excel.")
    return df_final

# llamamos a la función con el DataFrame optimizado de la lección 5
df_final = estructurar_final(df_optimizado)


#ejecución del flujo completo
if __name__ == "__main__":
    ids, montos = generar_datos_numpy()
    df_inicial = crear_dataframe(ids, montos)
    df_consolidado = integrar_fuentes(df_inicial)
    df_limpio = limpiar_datos(df_consolidado)
    df_optimizado = data_wrangling(df_limpio)
    df_final = estructurar_final(df_optimizado)

Resumen por región:
         promedio_monto  total_monto  cantidad_tx
region                                          
Este      54563.666667      1309528           24
Norte     49090.800000      1227270           25
Oeste     49724.640000     11188044          225
Sur       52993.461538      1377830           26

 Resumen por categoría de gasto:
                  promedio_monto  total_monto  cantidad_tx
categoria_gasto                                          
Bajo               18279.577778      1645162           90
Medio              49867.526316      5684898          114
Alto               80964.708333      7772612           96

 Tabla pivote (transacciones por región y categoría):
 categoria_gasto  Bajo  Medio  Alto
region                            
Este                6      6    12
Norte               6     16     3
Oeste              75     77    73
Sur                 3     15     8

 Formato largo con melt() (muestra):
    cliente_id tipo_monto    valor
0           0      mo

Documento resumen del flujo de trabajo completo (Lección 1 → Lección 6):  

**Lección 1 - NumPy:** Se generaron 300 IDs de clientes y montos de transacciones aleatorios usando distribución uniforme, redondeados a enteros. Se introdujo un outlier artificial en el índice 42 (monto=200.000) para ser detectado y eliminado posteriormente. Se calcularon métricas básicas (total, media, conteo). NumPy se utilizó por su eficiencia en operaciones numéricas vectorizadas, su integración con código compilado y su procesamiento paralelo.  

**Lección 2 - Pandas:** Los arrays de NumPy se convirtieron en un DataFrame con columnas nombradas. Se realizó exploración inicial (head, tail, describe) y se aplicó un filtro para transacciones mayores a $95.000. Se generaron 5 filas duplicadas artificialmente para ser eliminadas en la lección 5. Se exportó a CSV para servir de base a las siguientes etapas.  

**Lección 3 - Integración de fuentes:** Se cargó el CSV generado y se complementó con datos de región y puntos de fidelidad provenientes de una tabla HTML publicada en GitHub Pages. Para que read_html() funcionara fue necesario activar GitHub Pages en el repositorio y usar la URL correcta del sitio (no la URL de vista del repositorio ni la raw). Las fuentes se unificaron con merge() sobre cliente_id usando join tipo 'left' para no perder registros.  

**Lección 4 - Limpieza:** Se identificaron y trataron valores nulos (imputación por media para numéricos, por moda para categóricos). Los 201 nulos en 'region' y 'puntos_fidelidad' corresponden a los clientes con id >= 100, que no tienen registro en la tabla web. Se detectó y eliminó el outlier de monto=200.000 usando el método IQR.  

**Lección 5 - Data Wrangling:** Se eliminaron los 4 duplicados generados en la lección 2 (uno de los 5 ya había sido removido al eliminar el outlier). Se convirtieron los tipos de dato (monto a int). Se crearon columnas enriquecidas: monto con IVA del 19% (apply + lambda), clasificación sobre/bajo promedio (map + lambda), y categoría de gasto (pd.cut).  

**Lección 6 - Agrupamiento y pivoteo:** Se calcularon métricas resumidas por región y categoría con groupby() usando observed=True para evitar warnings de versiones futuras de pandas. Se generó una tabla pivote con pivot_table(). Se demostró la reestructuración a formato largo con melt() y la unión de un lote adicional de 20 clientes con concat(). El dataset final se exportó en CSV y Excel, listo para análisis o modelos predictivos.